In [ ]:
path = r".\ames.csv"

In [ ]:
# Ames House Price Analysis

## 1. Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
## 2. Loading the Dataset

In [ ]:
df = pd.read_csv(path)

FileNotFoundError: [Errno 2] No such file or directory: '.\\ames.csv'

In [ ]:
## 3. Initial Data Inspection

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
## 4. Missing Values Analysis

In [ ]:
missing = df.isna().sum()
missing[missing>0].sort_values(ascending = False)

In [ ]:
df["Misc_Feature"].value_counts(dropna = False)

In [ ]:
df["Mas_Vnr_Type"].value_counts(dropna = False)

In [ ]:
df.isna().sum().sum()

In [ ]:
missing

In [ ]:
missing_percent = (df.isna().sum() / len(df) )* 100

missing_percent[missing_percent > 0].sort_values(ascending =False)

In [ ]:
## 4. Missing Values Analysis

Missing values in `Misc_Feature` and `Mas_Vnr_Type` were replaced with `"None"` because the missing value represents the absence of that feature.


In [ ]:
df_clean = df.copy()

In [ ]:
df_clean["Misc_Feature"] = df_clean["Misc_Feature"].fillna("None")
df_clean["Mas_Vnr_Type"] = df_clean["Mas_Vnr_Type"].fillna("None")

In [ ]:
df_clean.isna().sum()[df_clean.isna().sum()>0]

In [ ]:
## 6. Basic Data Quality Checks

In [ ]:
df_clean.duplicated().sum()

In [ ]:
df_clean.dtypes.value_counts()

In [ ]:
df_clean.describe()

In [ ]:
## 7. Handling Invalid Values in Lot_Frontage

In [ ]:
(df_clean["Lot_Frontage"]==0).sum()

In [ ]:
df_clean[df_clean["Lot_Frontage"]==0][["Lot_Frontage","Lot_Area" ,"Lot_Shape" ,"Neighborhood" ,"Sale_Price"]].head(10)

In [ ]:
df_clean[df_clean["Lot_Frontage"] ==0]["Neighborhood"].value_counts()

In [ ]:
df_clean.loc[df_clean["Lot_Frontage"] == 0 , "Lot_Frontage"] = np.nan

# Convert zero values to NaN because a lot frontage of 0 is treated as a missing value.

In [ ]:
df_clean["Lot_Frontage"].isna().sum()

In [ ]:
df_clean["Lot_Frontage"].describe()

In [ ]:
df_clean["Lot_Frontage"] = df_clean["Lot_Frontage"].fillna(df_clean["Lot_Frontage"].median())

# Fill missing Lot_Frontage values with the median.

In [ ]:
## 8. Data Consistency Checks

In [ ]:
(df_clean["Sale_Price"] <= 0).sum()

In [ ]:
(df_clean["Lot_Area"] <= 0).sum()

In [ ]:
(df_clean["Year_Built"] > df_clean["Year_Sold"]).sum()

In [ ]:
df_clean.loc[df_clean["Year_Built"] > df_clean["Year_Sold"] , ["Year_Built" ,"Year_Sold" ,"Sale_Price" ,"Neighborhood"]]

In [ ]:
df_clean[df_clean["Neighborhood"] =="Edwards"][["Year_Built" ,"Year_Sold" ,"Sale_Price"]].describe()

In [ ]:
## 9. Sale Price Distribution

In [ ]:
df_clean["Sale_Price"].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
df_clean["MS_Zoning"].value_counts()

In [ ]:
## 10. Feature Engineering

In [ ]:
### 10.1 House Age

In [ ]:
df_clean["House_age"] = df_clean["Year_Sold"] - df_clean["Year_Built"]

In [ ]:
df_clean[["House_age" , "Year_Sold" ,"Year_Built"]].head()

In [ ]:
### 10.2 Price per Square Foot

In [ ]:
df_clean["Price_Per_Sq_Ft"] = df_clean["Sale_Price"] / df_clean["Gr_Liv_Area"]

In [ ]:
df_clean[["Price_Per_Sq_Ft" ,"Sale_Price" ,"Gr_Liv_Area"]].head()

In [ ]:
### 10.3 Converting Overall Quality to Numeric Values

In [ ]:
df_clean["Overall_Qual"].dtypes

In [ ]:
df_clean["Overall_Qual"].value_counts()

In [ ]:
quality_map = {
    "Very_Poor": 1,
    "Poor": 2,
    "Fair": 3,
    "Below_Average": 4,
    "Average": 5,
    "Above_Average": 6,
    "Good": 7,
    "Very_Good": 8,
    "Excellent": 9,
    "Very_Excellent": 10
}

In [ ]:
df_clean["Overall_Qual_Num"] = df_clean["Overall_Qual"].map(quality_map)

In [ ]:
### 10.3 Converting Overall Quality to Numeric Values

In [ ]:
df_clean["Luxury_House"] = np.where(df_clean["Overall_Qual_Num"] >= 8 , True ,False)

In [ ]:
df_clean[["Overall_Qual" ,"Overall_Qual_Num"]]

In [ ]:
df_clean["Luxury_House"].value_counts()

In [ ]:
### 10.5 Price Category

In [ ]:
conditions = [ 
    df_clean["Sale_Price"] < 100000 ,
    df_clean["Sale_Price"] < 200000 , 
    df_clean["Sale_Price"] < 300000 
]

choices = [
    "Budget" , "Mid_Range" , "High"]

df_clean["Price_Category"] = np.select(
    conditions , choices , default = "Luxury")

In [ ]:
df_clean["Price_Category"].value_counts()

In [ ]:
### 10.6 Total Bathrooms

In [ ]:
df_clean["Total_Bathrooms"] = (df_clean["Half_Bath"] * 0.5 + df_clean["Full_Bath"])

In [ ]:
df_clean[["Total_Bathrooms" ,"Half_Bath" ,"Full_Bath"]]

In [ ]:
### 10.7 Renovation Status

In [ ]:
df_clean["Is_Renovated"] = (df_clean["Year_Remod_Add"] > df_clean["Year_Built"])

In [ ]:
df_clean["Is_Renovated"].head()

In [ ]:
df_clean["Is_Renovated"].value_counts()

In [ ]:
## 11. Exploratory Data Analysis

In [ ]:
df_clean["Sale_Price"].mean()

In [ ]:
df_clean["Sale_Price"].median()

In [ ]:
### 11.1 Sale Price by Neighborhood

In [ ]:
df_clean.groupby("Neighborhood")["Sale_Price"].mean()

In [ ]:
df_clean.groupby("Neighborhood")["Sale_Price"].agg(["mean" ,"count"]).sort_values("mean" ,ascending = False)

In [ ]:
### 11.2 Living Area and Sale Price

In [ ]:
df_clean[["Gr_Liv_Area" , "Sale_Price"]].head(10)

In [ ]:
plt.scatter(df_clean["Gr_Liv_Area"] , df_clean["Sale_Price"])
plt.xlabel("Gr_Liv_Area")
plt.ylabel("Sale_Price")
plt.show()

### Insight

Larger living areas are generally associated with higher sale prices, although the relationship is not perfectly linear.

In [ ]:
df_clean["Gr_Liv_Area"].corr(df_clean["Sale_Price"])

In [ ]:
### 11.3 Overall Quality and Sale Price

In [ ]:
df_clean["Sale_Price"].corr(df_clean["Overall_Qual_Num"])

In [ ]:
plt.scatter(df_clean["Overall_Qual_Num"] ,df_clean["Sale_Price"])
plt.xlabel("Overall_Qual_Num")
plt.ylabel("Sale_Price")
plt.title("Overall Quality vs Sale Price")
plt.show()

### Insight

Higher overall quality is strongly associated with higher sale prices.

In [ ]:
### 11.4 House Age and Sale Price

In [ ]:
df_clean["House_age"].corr(df_clean["Sale_Price"])

In [ ]:
### 11.4 House Age and Sale Price

In [ ]:
df_clean["TotRms_AbvGrd"].corr(df_clean["Sale_Price"])

In [ ]:
### 11.4 House Age and Sale Price

In [ ]:
yearly_price = df_clean.groupby("Year_Sold")["Sale_Price"].mean()

In [ ]:
yearly_price

In [ ]:
plt.plot(yearly_price.index , yearly_price.values ,marker = "o")
plt.xlabel("Year Sold")
plt.ylabel("Average Sale Price ($)")
plt.title("Average Sale Price by Year")

plt.show()

### Insight

Average sale prices were highest in 2007 and lowest in 2010, with some fluctuations across the years.

In [ ]:
### 11.7 Average Sale Price by Number of Rooms

In [ ]:
room_price = df_clean.groupby("TotRms_AbvGrd")["Sale_Price"].agg(["mean" ,"count"])

In [ ]:
plt.plot(room_price.index, room_price["mean"], marker="o")
plt.xlabel("Total Rooms Above Ground")
plt.ylabel("Average Sale Price")
plt.title("Average Sale Price by Number of Rooms")
plt.show()

### Insight

Average sale price generally increases as the number of rooms increases, although the trend becomes less consistent for homes with very large numbers of rooms.

In [ ]:
room_price

In [ ]:
### 11.8 Average Sale Price by Overall Quality

In [ ]:
df_clean.groupby("Overall_Qual_Num")["Sale_Price"].agg(["mean" ,"count"])

In [ ]:
### 11.9 Price per Square Foot by Neighborhood

In [ ]:
df_clean.groupby("Neighborhood")["Price_Per_Sq_Ft"].agg(["mean" ,"count"]).sort_values("mean" , ascending = False)

In [ ]:
## 12. Outlier Analysis

In [ ]:
### 12.1 Extreme Price per Square Foot

In [ ]:
df_clean["Sale_Price"].max()

In [ ]:
df_clean.nlargest(10 ,"Price_Per_Sq_Ft")[["Price_Per_Sq_Ft" ,"Sale_Price" ,"Gr_Liv_Area" ,"Overall_Qual_Num" ,"Neighborhood"]]

In [ ]:
### 12.2 Investigating Unusual Properties

In [ ]:
df_clean.nsmallest(10 ,"Price_Per_Sq_Ft")[["Price_Per_Sq_Ft" ,"Sale_Price" ,"Gr_Liv_Area" ,"Overall_Qual_Num" ,"Neighborhood"]]

In [ ]:
df_clean["Gr_Liv_Area"].mean()

In [ ]:
df_clean["Gr_Liv_Area"].median()

In [ ]:
df_clean["Gr_Liv_Area"].max()

In [ ]:
df_clean.loc[df_clean["Gr_Liv_Area"] == 5642]

In [ ]:
df_clean[df_clean["Overall_Qual_Num"]==10][["Sale_Price" ,"Price_Per_Sq_Ft" ,"Neighborhood" ,"Gr_Liv_Area"]].sort_values("Gr_Liv_Area" ,ascending = False)

In [ ]:
df_clean[(df_clean["Overall_Qual_Num"]==10) & (df_clean["Neighborhood"]== "Edwards")][["Sale_Price" ,"Price_Per_Sq_Ft" ,"Neighborhood" ,"Gr_Liv_Area" ,"Year_Built" ,"Year_Sold" ]].sort_values("Gr_Liv_Area" ,ascending = False)

In [ ]:
df_clean[(df_clean["Overall_Qual_Num"]==10) & (df_clean["Neighborhood"]== "Edwards")].T

In [ ]:
df_clean.loc[(df_clean["Overall_Qual_Num"]==10) & (df_clean["Neighborhood"]== "Edwards"),["Sale_Price" ,"Price_Per_Sq_Ft" ,"Neighborhood" ,"Gr_Liv_Area" ,"Year_Built" ,"Year_Sold" ]]

In [ ]:
df_clean[df_clean["Year_Sold"] < df_clean["Year_Built"]]

In [ ]:
df_clean.loc[df_clean.index.isin([1498 ,2180 ,2181]) ,["Sale_Price" ,"Sale_Condition"]]


The cleaned dataset was loaded into a SQLite database to perform SQL-based analysis.

In [ ]:
## 13. SQL Analysis

In [ ]:
import sqlite3

conn = sqlite3.connect("house_prices.db")

df_clean.to_sql("houses", conn, if_exists="replace", index=False)

In [ ]:
### 13.1 Top Neighborhoods by Average Sale Price

In [ ]:
query = """
SELECT
    Neighborhood,
    AVG(Sale_Price) AS Avg_Sale_Price
FROM houses
GROUP BY Neighborhood
ORDER BY Avg_Sale_Price DESC
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

result

In [ ]:
### 13.2 Most Expensive Properties

In [ ]:
query = """
SELECT
    Neighborhood,
    Sale_Price
FROM houses
ORDER BY Sale_Price desc
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

result

In [ ]:
### 13.3 Low-Priced Properties with High Overall Quality

In [ ]:
query = """
SELECT Overall_Qual_Num , Sale_Price
FROM houses 
WHERE Overall_Qual_Num >= 8
ORDER BY Sale_Price asc
LIMIT 10 
"""

result = pd.read_sql_query(query , conn)

result

In [ ]:
### 14. Key Insights

# 14. Key Insights

- Overall quality has the strongest positive relationship with Sale Price among the variables analyzed.
- Larger living areas are generally associated with higher sale prices.
- House age has a negative relationship with Sale Price, meaning older houses tend to sell for lower prices.
- Sale prices vary considerably across neighborhoods, with Northridge, Stone_Brook, and Northridge_Heights among the highest-priced neighborhoods.
- The average sale price generally increases as the number of rooms increases, although the relationship becomes less consistent for homes with very large numbers of rooms.
- The analysis also identified unusual properties, including high-quality houses with unusually low prices and one record where the sale year is earlier than the construction year.

# 15. Conclusion

This project explored the Ames housing dataset using Python, Pandas, NumPy, and SQL. 
The analysis included data cleaning, feature engineering, exploratory data analysis, and identification of unusual observations.